In [1]:
import pandas as pd
import numpy as np
import folium
import re

In [2]:
# Read and preprocess data
file_path = r"D:\OVGU\AMS project\GIT\AMS_ShipTraj\physics based model\Rheinhafen Data.xlsx"
df = pd.read_excel(file_path, header=None)
df['combined'] = df.astype(str).agg(' '.join, axis=1)

In [3]:
# Utility function
def extract_field(pattern, text):
    match = re.search(pattern, text)
    return match.group(1) if match else None

# Extract fields from JSON strings
df['timestamp'] = df['combined'].apply(lambda x: extract_field(r'(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}\.\d+)', x))
df['lat'] = df['combined'].apply(lambda x: extract_field(r'"lat": ([\d\.]+)', x))
df['lon'] = df['combined'].apply(lambda x: extract_field(r'"lon": ([\d\.]+)', x))
df['speed'] = df['combined'].apply(lambda x: extract_field(r'"speed": ([\d\.]+)', x))
df['course'] = df['combined'].apply(lambda x: extract_field(r'"course": ([\d\.]+)', x))

# Convert types
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
df['lat'] = pd.to_numeric(df['lat'], errors='coerce')
df['lon'] = pd.to_numeric(df['lon'], errors='coerce')
df['speed'] = pd.to_numeric(df['speed'], errors='coerce')
df['course'] = pd.to_numeric(df['course'], errors='coerce')

# Drop rows with essential missing data
df = df.dropna(subset=['lat', 'lon', 'speed', 'course']).reset_index(drop=True)

In [29]:
def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # km
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(dlambda/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))  # in km

# --- Start Point ---
start_point = {
    "lat": 48.9798,
    "lon": 8.2598
}

# --- Find nearest point >= 300m away ---
df["distance_km"] = df.apply(lambda row: haversine(start_point["lat"], start_point["lon"], row["lat"], row["lon"]), axis=1)
eligible_df = df[df["distance_km"] >= 0.3]

if eligible_df.empty:
    raise ValueError("No AIS point at least 300 meters away found.")

nearest_idx = eligible_df["distance_km"].idxmin()
nearest_point = df.loc[nearest_idx]

# --- Plot on folium map ---
m = folium.Map(location=[start_point["lat"], start_point["lon"]], zoom_start=14)

# Line from start to AIS point
folium.PolyLine(
    locations=[
        [start_point["lat"], start_point["lon"]],
        [nearest_point["lat"], nearest_point["lon"]]
    ],
    color='orange',
    weight=4,
    tooltip=f"Start → Nearest AIS ({nearest_point['distance_km']*1000:.0f} m)"
).add_to(m)

# Markers
folium.Marker([start_point["lat"], start_point["lon"]], tooltip="Start", icon=folium.Icon(color="red")).add_to(m)
folium.Marker([nearest_point["lat"], nearest_point["lon"]], tooltip="Nearest AIS", icon=folium.Icon(color="green")).add_to(m)


m